# VWAP 1m vs 5m Leak-Free Comparison


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import Markdown, display
import pandas as pd

root = Path.cwd().resolve()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if not (root / "pyproject.toml").exists():
    raise RuntimeError("Could not locate repository root.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 160)


In [2]:
OUTPUT_DIR = Path(r"D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\vwap_timeframe_comparison_final_20260324")
print(OUTPUT_DIR)


D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\vwap_timeframe_comparison_final_20260324


## 1) Methodology


In [3]:
verdict = json.loads((OUTPUT_DIR / "final_verdict.json").read_text(encoding="utf-8"))
display(Markdown("\n".join([f"- {note}" for note in verdict["methodology_notes"]])))


- Semantique leak-free conservee: signal sur information disponible au bar t-1, execution au next open du timeframe teste.
- Le 5m est derive du 1m par resampling left-closed/left-labeled, sans usage d'information future.
- Le VWAP 5m reutilise la somme des price*volume sous-jacents 1m pour eviter une distortion artificielle du calcul de VWAP.
- Seuls les parametres exprimes en nombre de barres ont ete rescalés minimalement pour conserver des horizons temporels comparables, sans retuning des seuils, buffers, fenetres horaires, ni couts.

## 2) Timeframe-Specific Variant Specs


In [4]:
display(pd.read_csv(OUTPUT_DIR / "variant_timeframe_specs.csv"))


,timeframe,bar_minutes,display_order,strategy_id,family,mode,execution_profile,quantity_mode,slope_lookback,atr_period,compression_length,pullback_lookback,atr_buffer,stop_buffer,max_trades_per_day,time_windows,what_changes_vs_baseline
0,1m,1,1,paper_vwap_baseline,baseline,target_position,paper_reference,paper_full_notional,5,14,3,6,0.25,0.25,NaN,full_rth,"Reference officielle: close[t-1] vs VWAP[t-1],..."
1,1m,1,2,baseline_futures_adapted,baseline,target_position,repo_realistic,fixed_quantity,5,14,3,6,0.25,0.25,NaN,full_rth,"Meme signal que la baseline paper, mais sous s..."
2,1m,1,3,vwap_reclaim,prop_variant,discrete,repo_realistic,fixed_quantity,5,14,4,6,0.25,0.25,3.0,full_rth,Variante discrete reclaim simple avec pente VW...
3,5m,5,1,paper_vwap_baseline,baseline,target_position,paper_reference,paper_full_notional,1,3,2,2,0.25,0.25,NaN,full_rth,"Reference officielle: close[t-1] vs VWAP[t-1],..."
4,5m,5,2,baseline_futures_adapted,baseline,target_position,repo_realistic,fixed_quantity,1,3,2,2,0.25,0.25,NaN,full_rth,"Meme signal que la baseline paper, mais sous s..."
5,5m,5,3,vwap_reclaim,prop_variant,discrete,repo_realistic,fixed_quantity,1,3,2,2,0.25,0.25,3.0,full_rth,Variante discrete reclaim simple avec pente VW...


## 3) Headline Comparison


In [5]:
display(pd.read_csv(OUTPUT_DIR / "comparison_summary.csv"))
display(pd.read_csv(OUTPUT_DIR / "comparison_delta.csv"))


,timeframe,bar_minutes,strategy_id,role,family,mode,execution_profile,what_changes_vs_baseline,overall_net_pnl,oos_net_pnl,oos_profit_factor,oos_sharpe_ratio,oos_max_drawdown,oos_expectancy_per_trade,oos_total_trades,pnl_nominal,pnl_slip_x2,pf_nominal,pf_slip_x2,sharpe_nominal,sharpe_slip_x2,dd_nominal,dd_slip_x2,delta_pnl_nominal_vs_slip_x2,pass_fail_cost_stress,positive_oos_splits,total_splits,mean_oos_net_pnl_splits,mean_oos_profit_factor_splits,mean_oos_sharpe_ratio_splits,worst_oos_split_net_pnl,best_oos_split_net_pnl,pass_fail_splits,worst_daily_loss_usd,daily_loss_limit_breach_freq,trailing_drawdown_breach_freq,avg_trades_per_day,max_trades_per_day,challenge_success_rate_standard,prop_verdict,top_3_day_contribution_pct,top_5_day_contribution_pct,pnl_excluding_top_3_days,pnl_excluding_top_5_days,concentration_verdict,pass_oos_nominal,pass_cost_stress,pass_splits,pass_prop,survives_primary_filter,reranking_score,final_bucket,elimination_reason,rank_within_survivors
0,1m,1,paper_vwap_baseline,paper_baseline_reference,baseline,target_position,paper_reference,"Reference officielle: close[t-1] vs VWAP[t-1],...",-25008.000000,-2376.500000,0.766288,-0.706673,-2518.500000,-3.876835,613,-2376.500000,-2989.500000,0.766288,0.720529,-0.706673,-0.860157,-2518.500000,-2991.000000,-613.0,False,0,4,-5460.125000,0.585349,-0.950730,-12137.500000,0.000000,False,-1380.000000,0.001905,0.940952,1.167619,53,0.0,non defendable,-0.731959,-0.937092,-4116.000000,-4603.500000,forte dependance aux meilleurs jours,False,False,False,False,False,0,reference_officielle,"Ancre officielle de comparaison, non candidate...",NaN
1,1m,1,baseline_futures_adapted,realistic_baseline_reference,baseline,target_position,repo_realistic,"Meme signal que la baseline paper, mais sous s...",-42674.500000,-20043.000000,0.900446,-1.412155,-20185.000000,-2.324096,8624,-20043.000000,-28667.000000,0.900446,0.862572,-1.412155,-1.986944,-20185.000000,-28668.500000,-8624.0,False,0,4,-22906.875000,0.893610,-1.521253,-29804.000000,-16787.500000,False,-1931.500000,0.011429,0.872381,16.426667,53,0.0,non defendable,-0.276580,-0.412039,-25586.500000,-28301.500000,forte dependance aux meilleurs jours,False,False,False,False,False,0,baseline_realiste_de_reference,Baseline economique de reference pour juger si...,NaN
2,1m,1,vwap_reclaim,candidate,prop_variant,discrete,repo_realistic,Variante discrete reclaim simple avec pente VW...,-416.758929,151.053571,1.140522,0.125315,-750.446429,7.950188,19,151.053571,132.053571,1.140522,1.121267,0.125315,0.109574,-750.446429,-764.446429,-19.0,True,1,4,-38.696429,0.973630,-0.029248,-117.946429,151.053571,False,-161.892857,0.000000,0.000000,0.036190,1,0.0,potentiellement compatible sous contraintes pr...,7.527131,8.116326,-985.946429,-1074.946429,forte dependance aux meilleurs jours,True,True,False,True,False,3,interessante mais trop fragile,Garde un signal partiel mais echoue sur les sp...,NaN
3,5m,5,paper_vwap_baseline,paper_baseline_reference,baseline,target_position,paper_reference,"Reference officielle: close[t-1] vs VWAP[t-1],...",-8172.500000,-3239.500000,0.978870,-0.231765,-7327.000000,-0.834708,3881,-3239.500000,-7120.500000,0.978870,0.954443,-0.231765,-0.505742,-7327.000000,-10312.500000,-3881.0,False,0,4,-5094.625000,0.970924,-0.323724,-9865.000000,-815.000000,False,-2065.500000,0.011429,0.638095,7.392381,23,0.0,non defendable,-2.046766,-2.849051,-9870.000000,-12469.000000,forte dependance aux meilleurs jours,False,False,False,False,False,0,reference_officielle,"Ancre officielle de comparaison, non candidate...",NaN
4,5m,5,baseline_futures_adapted,realistic_baseline_reference,baseline,target_position,repo_realistic,"Meme signal que la baseline paper, mais sous s...",-8172.500000,-3239.500000,0.978870,-0.231765,-7327.000000,-0.834708,3881,-3239.500000,-7120.500000,0.978870,0.954443,-0.231765,-0.505742,-7327.000000,-10312.500000,-3881.0,False,0,4,-5094.625000,0.970924,-0.323724,-9865.000000,-815.000000,False,-2065.500000,0.011429,0.638095,7

,strategy_id,role,oos_net_pnl_1m,oos_net_pnl_5m,oos_profit_factor_1m,oos_profit_factor_5m,oos_sharpe_ratio_1m,oos_sharpe_ratio_5m,oos_max_drawdown_1m,oos_max_drawdown_5m,oos_total_trades_1m,oos_total_trades_5m,expectancy_per_trade_1m,expectancy_per_trade_5m,pnl_slip_x2_1m,pnl_slip_x2_5m,positive_oos_splits_1m,positive_oos_splits_5m,challenge_success_rate_standard_1m,challenge_success_rate_standard_5m,top_5_day_contribution_pct_1m,top_5_day_contribution_pct_5m,pnl_excluding_top_5_days_1m,pnl_excluding_top_5_days_5m,trade_retention_ratio_5m_vs_1m,improvement_score,credible_robustness_improvement,comparison_bucket,comparison_comment,oos_and_costs_ok_5m,split_stability_improves,prop_metrics_improve,concentration_improves,trade_count_retained
0,paper_vwap_baseline,paper_baseline_reference,-2376.500000,-3239.500000,0.766288,0.978870,-0.706673,-0.231765,-2518.500000,-7327.000000,613,3881,-3.876835,-0.834708,-2989.500000,-7120.500000,0,0,0.0,0.0,-0.937092,-2.849051,-4603.500000,-12469.000000,6.331158,1,False,pas d'amelioration exploitable,Le passage en 5m ne produit pas d'amelioration...,False,False,False,False,True
1,baseline_futures_adapted,realistic_baseline_reference,-20043.000000,-3239.500000,0.900446,0.978870,-1.412155,-0.231765,-20185.000000,-7327.000000,8624,3881,-2.324096,-0.834708,-28667.000000,-7120.500000,0,0,0.0,0.0,-0.412039,-2.849051,-28301.500000,-12469.000000,0.450023,3,False,amelioration partielle mais insuffisante,"Le 5m lisse certains degats, mais pas assez po...",False,False,True,True,True
2,vwap_reclaim,candidate,151.053571,-1593.166667,1.140522,0.810555,0.125315,-0.567988,-750.446429,-2923.208333,19,116,7.950188,-13.734195,132.053571,-1709.166667,1,0,0.0,0.0,8.116326,-1.594937,-1074.946429,-4134.166667,6.105263,1,False,pas d'amelioration exploitable,Le passage en 5m ne produit pas d'amelioration...,False,False,False,False,True


## 4) Paper Baseline Reference


In [6]:
display(pd.read_csv(OUTPUT_DIR / "paper_baseline_reference_comparison.csv"))


,timeframe,bar_minutes,strategy_id,oos_net_pnl,oos_profit_factor,oos_sharpe_ratio,oos_max_drawdown,oos_total_trades,pnl_slip_x2,pf_slip_x2,positive_oos_splits,challenge_success_rate_standard,top_5_day_contribution_pct,final_bucket
0,1m,1,paper_vwap_baseline,-2376.5,0.766288,-0.706673,-2518.5,613,-2989.5,0.720529,0,0.0,-0.937092,reference_officielle
1,5m,5,paper_vwap_baseline,-3239.5,0.978870,-0.231765,-7327.0,3881,-7120.5,0.954443,0,0.0,-2.849051,reference_officielle


## 5) Stress x2


In [7]:
display(pd.read_csv(OUTPUT_DIR / "stress_test_summary.csv"))


,timeframe,bar_minutes,strategy_id,role,pnl_nominal,pnl_slip_x2,pf_nominal,pf_slip_x2,sharpe_nominal,sharpe_slip_x2,dd_nominal,dd_slip_x2,delta_pnl_nominal_vs_slip_x2,pass_fail_cost_stress
0,1m,1,paper_vwap_baseline,paper_baseline_reference,-2376.500000,-2989.500000,0.766288,0.720529,-0.706673,-0.860157,-2518.500000,-2991.000000,-613.0,False
1,1m,1,baseline_futures_adapted,realistic_baseline_reference,-20043.000000,-28667.000000,0.900446,0.862572,-1.412155,-1.986944,-20185.000000,-28668.500000,-8624.0,False
2,1m,1,vwap_reclaim,candidate,151.053571,132.053571,1.140522,1.121267,0.125315,0.109574,-750.446429,-764.446429,-19.0,True
3,5m,5,paper_vwap_baseline,paper_baseline_reference,-3239.500000,-7120.500000,0.978870,0.954443,-0.231765,-0.505742,-7327.000000,-10312.500000,-3881.0,False
4,5m,5,baseline_futures_adapted,realistic_baseline_reference,-3239.500000,-7120.500000,0.978870,0.954443,-0.231765,-0.505742,-7327.000000,-10312.500000,-3881.0,False
5,5m,5,vwap_reclaim,candidate,-1593.166667,-1709.166667,0.810555,0.798712,-0.567988,-0.608971,-2923.208333,-2971.208333,-116.0,False


## 6) Multi-Split


In [8]:
display(pd.read_csv(OUTPUT_DIR / "split_summary.csv"))


,timeframe,bar_minutes,strategy_id,positive_oos_splits,total_splits,mean_oos_net_pnl,mean_oos_profit_factor,mean_oos_sharpe_ratio,worst_oos_split_net_pnl,best_oos_split_net_pnl,pass_fail_splits
0,1m,1,paper_vwap_baseline,0,4,-5460.125000,0.585349,-0.950730,-12137.500000,0.000000,False
1,1m,1,baseline_futures_adapted,0,4,-22906.875000,0.893610,-1.521253,-29804.000000,-16787.500000,False
2,1m,1,vwap_reclaim,1,4,-38.696429,0.973630,-0.029248,-117.946429,151.053571,False
3,5m,5,paper_vwap_baseline,0,4,-5094.625000,0.970924,-0.323724,-9865.000000,-815.000000,False
4,5m,5,baseline_futures_adapted,0,4,-5094.625000,0.970924,-0.323724,-9865.000000,-815.000000,False
5,5m,5,vwap_reclaim,0,4,-1553.322917,0.822603,-0.527157,-1776.041667,-1303.916667,False


## 7) Prop Metrics


In [9]:
display(pd.read_csv(OUTPUT_DIR / "prop_summary.csv"))


,timeframe,bar_minutes,strategy_id,role,worst_daily_loss_usd,red_days_freq,days_below_neg_0p25r_freq,days_below_neg_0p5r_freq,days_below_neg_1r_freq,worst_losing_trades_streak,worst_losing_days_streak,max_drawdown,daily_loss_limit_breach_freq,trailing_drawdown_breach_freq,avg_trades_per_day,max_trades_per_day,challenge_success_rate_standard,challenge_bust_rate_standard,prop_verdict
0,1m,1,paper_vwap_baseline,paper_baseline_reference,-1380.000000,0.969524,NaN,NaN,NaN,51,4,-2518.500000,0.001905,0.940952,1.167619,53,0.0,0.0,non defendable
1,1m,1,baseline_futures_adapted,realistic_baseline_reference,-1931.500000,0.539048,NaN,NaN,NaN,52,8,-20185.000000,0.011429,0.872381,16.426667,53,0.0,0.0,non defendable
2,1m,1,vwap_reclaim,candidate,-161.892857,0.990476,0.022857,0.015238,0.011429,5,1,-750.446429,0.000000,0.000000,0.036190,1,0.0,0.0,potentiellement compatible sous contraintes pr...
3,5m,5,paper_vwap_baseline,paper_baseline_reference,-2065.500000,0.512381,NaN,NaN,NaN,28,7,-7327.000000,0.011429,0.638095,7.392381,23,0.0,0.0,non defendable
4,5m,5,baseline_futures_adapted,realistic_baseline_reference,-2065.500000,0.512381,NaN,NaN,NaN,28,7,-7327.000000,0.011429,0.638095,7.392381,23,0.0,0.0,non defendable
5,5m,5,vwap_reclaim,candidate,-617.666667,0.935238,0.106667,0.059048,0.015238,12,2,-2923.208333,0.000000,0.356190,0.220952,2,0.0,0.0,non defendable


## 8) Concentration


In [10]:
display(pd.read_csv(OUTPUT_DIR / "concentration_summary.csv"))


,timeframe,bar_minutes,strategy_id,role,top_3_day_contribution_pct,top_5_day_contribution_pct,pnl_excluding_top_3_days,pnl_excluding_top_5_days,pnl_excluding_best_month,concentration_verdict
0,1m,1,paper_vwap_baseline,paper_baseline_reference,-0.731959,-0.937092,-4116.000000,-4603.500000,-2376.500000,forte dependance aux meilleurs jours
1,1m,1,baseline_futures_adapted,realistic_baseline_reference,-0.276580,-0.412039,-25586.500000,-28301.500000,-22736.000000,forte dependance aux meilleurs jours
2,1m,1,vwap_reclaim,candidate,7.527131,8.116326,-985.946429,-1074.946429,-529.446429,forte dependance aux meilleurs jours
3,5m,5,paper_vwap_baseline,paper_baseline_reference,-2.046766,-2.849051,-9870.000000,-12469.000000,-6427.000000,forte dependance aux meilleurs jours
4,5m,5,baseline_futures_adapted,realistic_baseline_reference,-2.046766,-2.849051,-9870.000000,-12469.000000,-6427.000000,forte dependance aux meilleurs jours
5,5m,5,vwap_reclaim,candidate,-1.138927,-1.594937,-3407.666667,-4134.166667,-2705.666667,forte dependance aux meilleurs jours


## 9) Final Verdict


In [11]:
verdict = json.loads((OUTPUT_DIR / "final_verdict.json").read_text(encoding="utf-8"))
verdict
display(Markdown((OUTPUT_DIR / "comparison_summary.md").read_text(encoding="utf-8")))


# VWAP 1m vs 5m Comparison

- Global verdict: `5 minutes ne change pas le verdict, famille VWAP a archiver`
- 1m verdict: `Aucune variante n'est assez robuste pour meriter une validation approfondie supplementaire.`
- 5m verdict: `Aucune variante n'est assez robuste pour meriter une validation approfondie supplementaire.`
- Credible candidates in 5m: `none`

## Delta Table

```text
             strategy_id                         role  oos_net_pnl_1m  oos_net_pnl_5m  oos_profit_factor_1m  oos_profit_factor_5m  oos_sharpe_ratio_1m  oos_sharpe_ratio_5m  oos_max_drawdown_1m  oos_max_drawdown_5m  oos_total_trades_1m  oos_total_trades_5m  expectancy_per_trade_1m  expectancy_per_trade_5m  pnl_slip_x2_1m  pnl_slip_x2_5m  positive_oos_splits_1m  positive_oos_splits_5m  challenge_success_rate_standard_1m  challenge_success_rate_standard_5m  top_5_day_contribution_pct_1m  top_5_day_contribution_pct_5m  pnl_excluding_top_5_days_1m  pnl_excluding_top_5_days_5m  trade_retention_ratio_5m_vs_1m  improvement_score  credible_robustness_improvement                        comparison_bucket                                                            comparison_comment  oos_and_costs_ok_5m  split_stability_improves  prop_metrics_improve  concentration_improves  trade_count_retained
     paper_vwap_baseline     paper_baseline_reference    -2376.500000    -3239.500000              0.766288              0.978870            -0.706673            -0.231765         -2518.500000         -7327.000000                  613                 3881                -3.876835                -0.834708    -2989.500000    -7120.500000                       0                       0                                 0.0                                 0.0                      -0.937092                      -2.849051                 -4603.500000                -12469.000000                        6.331158                  1                            False           pas d'amelioration exploitable            Le passage en 5m ne produit pas d'amelioration robuste suffisante.                False                     False                 False                   False                  True
baseline_futures_adapted realistic_baseline_reference   -20043.000000    -3239.500000              0.900446              0.978870            -1.412155            -0.231765        -20185.000000         -7327.000000                 8624                 3881                -2.324096                -0.834708   -28667.000000    -7120.500000                       0                       0                                 0.0                                 0.0                      -0.412039                      -2.849051                -28301.500000                -12469.000000                        0.450023                  3                            False amelioration partielle mais insuffisante Le 5m lisse certains degats, mais pas assez pour renverser le verdict global.                False                     False                  True                    True                  True
            vwap_reclaim                    candidate      151.053571    -1593.166667              1.140522              0.810555             0.125315            -0.567988          -750.446429         -2923.208333                   19                  116                 7.950188               -13.734195      132.053571    -1709.166667                       1                       0                                 0.0                                 0.0                       8.116326                      -1.594937                 -1074.946429                 -4134.166667                        6.105263                  1                            False           pas d'amelioration exploitable            Le passage en 5m ne produit pas d'amelioration robuste suffisante.                False                     False                 False                   False                  True
```

## Comparison Summary

```text
timeframe              strategy_id                         role   oos_net_pnl  oos_profit_factor  oos_sharpe_ratio  oos_max_drawdown  oos_total_trades   pnl_slip_x2  positive_oos_splits  challenge_success_rate_standard  top_5_day_contribution_pct                   final_bucket
       1m      paper_vwap_baseline     paper_baseline_reference  -2376.500000           0.766288         -0.706673      -2518.500000               613  -2989.500000                    0                              0.0                   -0.937092           reference_officielle
       1m baseline_futures_adapted realistic_baseline_reference -20043.000000           0.900446         -1.412155     -20185.000000              8624 -28667.000000                    0                              0.0                   -0.412039 baseline_realiste_de_reference
       1m             vwap_reclaim                    candidate    151.053571           1.140522          0.125315       -750.446429                19    132.053571                    1                              0.0                    8.116326 interessante mais trop fragile
       5m      paper_vwap_baseline     paper_baseline_reference  -3239.500000           0.978870         -0.231765      -7327.000000              3881  -7120.500000                    0                              0.0                   -2.849051           reference_officielle
       5m baseline_futures_adapted realistic_baseline_reference  -3239.500000           0.978870         -0.231765      -7327.000000              3881  -7120.500000                    0                              0.0                   -2.849051 baseline_realiste_de_reference
       5m             vwap_reclaim                    candidate  -1593.166667           0.810555         -0.567988      -2923.208333               116  -1709.166667                    0                              0.0                   -1.594937         eliminee immediatement
```
